# Day 35 — Confidence Calibration & Abstention
Fit calibrator và threshold từ OOF training-subject scores only.


In [ ]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
OOF_SCORES=Path('/content/drive/MyDrive/MyoLab-AI-data/day35/oof-scores.npz')
VALIDATION_SCORES=Path('/content/drive/MyDrive/MyoLab-AI-data/day35/frozen-validation-scores.npz')
OUTPUT_DIR=Path('/content/drive/MyDrive/MyoLab-AI-data/day35/output');OUTPUT_DIR.mkdir(parents=True,exist_ok=True)


## Bước 1 — Score/partition gate


In [ ]:
oof=np.load(OOF_SCORES,allow_pickle=False);val=np.load(VALIDATION_SCORES,allow_pickle=False)
required={'scores','y_index','subject_ids','class_order'}
assert not (required-set(oof.files));assert not (required-set(val.files))
assert not (set(oof['subject_ids'].tolist()) & set(val['subject_ids'].tolist())), 'OOF_VALIDATION_SUBJECT_OVERLAP'


## Bước 2 — Fit calibrator trên OOF only


In [ ]:
import sys
sys.path.insert(0,str(Path.cwd()/'ai-core'/'calibration'))
from day35.calibration import TemperatureScaler
from day35.metrics import multiclass_brier,expected_calibration_error,coverage_risk
calibrator=TemperatureScaler().fit(oof['scores'],oof['y_index'])
oof_proba=calibrator.predict_proba(oof['scores'])


## Bước 3 — Freeze rồi evaluate validation


In [ ]:
val_proba=calibrator.predict_proba(val['scores'])
ece,bins=expected_calibration_error(val['y_index'],val_proba,15)
metrics={'brier':multiclass_brier(val['y_index'],val_proba),'ece':ece,'temperature':calibrator.temperature_,'fit_partition':'OOF_TRAINING_SUBJECTS_ONLY'}
print(metrics)


## Bước 4 — Threshold chọn trên OOF only


In [ ]:
thresholds=np.linspace(0,1,101)
curve=coverage_risk(oof['y_index'],oof_proba,thresholds)
eligible=[r for r in curve if r['coverage']>=.8 and r['selective_risk']<=.1]
threshold=max(r['threshold'] for r in eligible) if eligible else 1.0
print('Frozen threshold',threshold,coverage_risk(val['y_index'],val_proba,[threshold]))
joblib.dump(calibrator,OUTPUT_DIR/'calibration-model.joblib')
(OUTPUT_DIR/'calibration-metrics.json').write_text(json.dumps(metrics,indent=2))
